In [0]:
# This step is done so it is commented out
# dbutils.fs.mkdirs("dbfs:/FileStore/iot_krish/raw/")

In [0]:
# Load secrets from Databricks Secret Scope
API_KEY = dbutils.secrets.get(scope="iot_pipeline_krish", key="api_key")
STORAGE_KEY = dbutils.secrets.get(scope="iot_pipeline_krish", key="storage_key")
WEBHOOK_URL = dbutils.secrets.get(scope="iot_pipeline_krish", key="webhook_url")

print("Secrets loaded successfully")

In [0]:
%sql
CREATE CATALOG IF NOT EXISTS iot_krish;

CREATE SCHEMA IF NOT EXISTS iot_krish.bronze;
CREATE SCHEMA IF NOT EXISTS iot_krish.silver;
CREATE SCHEMA IF NOT EXISTS iot_krish.gold;

In [0]:
from pyspark.sql.functions import *
from pyspark.sql.types import *

In [0]:
sensor_path = "dbfs:/FileStore/iot/raw/sensor_readings.json"
sensor_df = spark.read.option("multiLine", True).option("inferSchema", True).json(sensor_path)
sensor_df.printSchema()
display(sensor_df)

In [0]:
sensor_df = spark.read.option("inferSchema", True).json(sensor_path)

In [0]:
sensor_bronze_df = (
    sensor_df
    .withColumn("_ingested_at", current_timestamp())
    .withColumn("_source_file", lit("sensor_readings.json"))
)

In [0]:
(sensor_bronze_df.write
 .format("delta")
 .mode("overwrite")
 .saveAsTable("iot_krish.bronze.sensor_readings"))

In [0]:
devices_path = "dbfs:/FileStore/iot/raw/devices.json"

devices_df = spark.read.json(devices_path)

(devices_df.write
 .format("delta")
 .mode("overwrite")
 .saveAsTable("iot_krish.bronze.devices"))

In [0]:
locations_path = "dbfs:/FileStore/iot/raw/locations.json"

locations_df = spark.read.json(locations_path)

(locations_df.write
 .format("delta")
 .mode("overwrite")
 .saveAsTable("iot_krish.bronze.locations"))

In [0]:
%sql
ALTER TABLE iot_krish.bronze.sensor_readings
DROP CONSTRAINT IF EXISTS valid_temperature; ALTER TABLE iot_krish.bronze.sensor_readings ADD CONSTRAINT valid_temperature
CHECK (temperature_c >= -50 AND temperature_c <= 200);

In [0]:
bad_df = sensor_bronze_df.withColumn("temperature_c", col("temperature_c").cast("string"))

try:
    (bad_df.write
     .format("delta")
     .mode("append")
     .saveAsTable("iot_krish.bronze.sensor_readings"))
except Exception as e:
    print("Expected schema enforcement error:")
    print(e)

## Silver Layer

In [0]:
sensor_bronze_df_path = "iot_krish.bronze.sensor_readings"
devices_bronze_df_path = "iot_krish.bronze.devices"
locations_bronze_df_path = "iot_krish.bronze.locations"

sensor_bronze = spark.table(sensor_bronze_df_path)
devices_bronze = spark.table(devices_bronze_df_path)
locations_bronze = spark.table(locations_bronze_df_path)

In [0]:
sensor_clean_df = (
    sensor_bronze
    .withColumn("reading_ts", to_timestamp("reading_ts"))
    .withColumn("reading_date", to_date("reading_ts"))
)

In [0]:
sensor_clean_df = sensor_clean_df.withColumn(
    "is_anomaly",
    when((col("temperature_c") > 90) | (col("vibration_hz") > 3.0), True).otherwise(False)
)

In [0]:
from pyspark.sql.window import Window

rolling_window = Window.partitionBy("device_id").orderBy("reading_ts").rowsBetween(-2, 0)

sensor_clean_df = sensor_clean_df.withColumn(
    "rolling_avg_temp",
    avg("temperature_c").over(rolling_window)
)

In [0]:
sensor_clean_join_df = sensor_clean_df \
    .join(devices_bronze, "device_id", "left") \
    .join(locations_bronze, "location_id", "left") \
    .select(
        sensor_clean_df["*"],
        devices_bronze["device_type"],
        devices_bronze["maintenance_due"],
        locations_bronze["zone_name"],
        locations_bronze["floor"],
        locations_bronze["supervisor"]
    )

In [0]:
display(sensor_clean_join_df)

In [0]:
(sensor_clean_join_df.write
 .format("delta")
 .mode("overwrite")
 .partitionBy("location_id")
 .option("overwriteSchema", "true")
 .saveAsTable("iot_krish.silver.sensor_readings"))

In [0]:
%sql
OPTIMIZE iot_krish.silver.sensor_readings
ZORDER BY (device_id, reading_ts);

# Gold Layer

In [0]:
%sql
CREATE OR REPLACE VIEW iot_krish.gold.hourly_device_health AS
SELECT
    device_id,
    DATE_TRUNC('hour', reading_ts) AS reading_hour,
    AVG(temperature_c) AS avg_temperature,
    AVG(vibration_hz) AS avg_vibration,
    SUM(CASE WHEN is_anomaly THEN 1 ELSE 0 END) AS anomaly_count
FROM iot_krish.silver.sensor_readings
GROUP BY
    device_id,
    DATE_TRUNC('hour', reading_ts);

In [0]:
%sql
CREATE OR REPLACE VIEW iot_krish.gold.critical_alerts AS
SELECT *
FROM iot_krish.silver.sensor_readings
WHERE lower(status) = 'critical'
ORDER BY reading_ts DESC;